In [1]:
# !pip install youtube-transcript-api

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
from langchain_ollama import OllamaEmbeddings

In [5]:
from langchain_community.vectorstores import FAISS

C:\Users\HP\AppData\Local\Temp\ipykernel_6760\1059762981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
from langchain_core.prompts import PromptTemplate

Document loader

In [14]:
yt_api = YouTubeTranscriptApi()


In [15]:
video_id = 'SfOaZIGJ_gs'
transcripts = yt_api.fetch(video_id)

In [16]:
# transcripts

In [17]:
transcript_text = " ".join(doc.text for doc in transcripts)

In [18]:
# transcript_text

In [19]:
splitter =  RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)

In [20]:
chunks = splitter.split_text(transcript_text)

In [21]:
# chunks

In [22]:
print(len(chunks))

88


In [23]:
# Embedding model 

In [24]:
Emeddings = OllamaEmbeddings(model='nomic-embed-text')

In [25]:
vector_store = FAISS.from_texts(chunks,Emeddings)

In [26]:
vector_store

In [27]:
vector_store.similarity_search(query="Tell me about AI")

[Document(id='0e6d6f9a-ba63-4990-b1dc-966298ea6d9f', metadata={}, page_content="I one one of the things that I think we'll be defining about the difference of AI versus the sort of previous way we've been using computers and technology is you really want AI to be have as much context as possible do stuff for you and be proactive. So, a computer or a phone, you know, it's kind of either on or off. It's in your pocket or it's in your hand and you're using it. But you might want AI to just be, you know, like a companion with you throughout your day and alerting you in"),
 Document(id='167afa95-666a-413e-8eb2-846abf189c18', metadata={}, page_content="about what AI is going to mean for science and the amount of science that one person will be able to discover and the rate at which they can do that. Clearly, it's transforming what it means to program computers in a huge way and people will be able to create completely new kinds of software in a at huge new new scale. Uh definitely for startu

CREATING RETRIEVER

In [28]:
retriever = vector_store.as_retriever(search_kwargs = {"k":4})

In [29]:
# context 
retriever.invoke("How to start a company")

[Document(id='7bde0bf5-81b5-4da4-b823-fec5b11b14b5', metadata={}, page_content="that match that form factor. It seems like a good idea. >> If I'm a young guy looking to start a robotics company, but somebody else has manufacturing scale and I really want as a Indian guy to be able to build and compete there. How do I make up for manufacturing scale as someone starting up? >> Well, eventually once you build enough robots, they can make more copies of themselves. And uh but in the short term I think you probably have to find some really good partners that know a lot about"),
 Document(id='b5766748-be82-4941-901e-3f8e36fbbf2d', metadata={}, page_content="relevance they might have had when I was growing up. But what do I do now? A what do I study? If I'm starting a company, what kind of company do I start? Or if I were to even find a job, what industry do you think has some kind of tailwind? I'm not talking 10 years down the line, but even as close as three to five years down the line. >> 

Augmentation 

In [30]:
prompt = PromptTemplate(template=""" You are a helpful assistent . Answer from the following context. 
               if context is insufficient Then Directly say I dont know 
               {context}
               Question {question}""",
               input_variables=['context','question'])

In [31]:
question = "How do Sam altman and Nikhil kamath describe their own learning journey with AI in this video"

In [32]:
context = retriever.invoke(question)

In [33]:
# context

In [34]:
context = " ".join(i.page_content for i in context)

In [35]:
final_prompt = prompt.invoke({'question':question, 'context':context})

Generation model 

In [36]:
from langchain_ollama.chat_models import ChatOllama

In [37]:
llm = ChatOllama(model='gemma3', temperature=0.7)

In [ ]:
Answer = llm.invoke(final_prompt)


In [41]:
print(Answer.content)

According to the context, Sam Altman and Nikhil Kamath describe their own learning journey with AI as being “really good at using the new AI tools.” They emphasize the value of learning to learn and highlight AI tools as a “meta skill” that will be valuable across various fields like engineering and biology. They also suggest that learning to program was a valuable skill to acquire in the past, and currently, fluency with AI tools is the most important specific hard skill to learn.
